# QLoRA fine-tuning on Kaggle free GPUDeepSeek-R1-Distill-Qwen-1.5B + 4-bit NF4 QLoRA + SFT, on a personal-profile QA dataset.**Before you run this**1. Settings -> Accelerator -> **GPU T4 x2** (or P100). Anything without CUDA will fail at the training step.2. Settings -> Internet -> **On** (needed to pip-install and to download the base model).3. Add-ons -> Secrets -> add `HF_TOKEN` **only if** you want to push the adapter to the Hub. Training works without it.4. Upload your `personal_dataset.jsonl` as a **private Kaggle Dataset** and attach it, or paste it in the cell marked *Dataset*.Runtime is roughly 25-45 minutes for 5 epochs on ~620 examples on a T4.Nothing in this notebook prints raw personal answers.

## 1. Environment: install dependencies

In [ ]:
!pip install -q -U "transformers>=4.51.0" "trl>=0.17.0" "peft>=0.14.0" \    "bitsandbytes>=0.45.0" "accelerate>=1.2.0" "datasets>=3.0.0" "huggingface_hub>=0.28.0"print("installed")

## 2. Verify the GPUStop here if this cell reports no CUDA device - QLoRA cannot run on CPU.

In [ ]:
import torch, subprocessassert torch.cuda.is_available(), (    "No CUDA GPU. Set Settings -> Accelerator -> GPU T4 x2 and restart the session.")name = torch.cuda.get_device_name(0)total = torch.cuda.get_device_properties(0).total_memory / 1024**3free, _ = torch.cuda.mem_get_info()print(f"GPU              : {name}")print(f"Total memory     : {total:.1f} GiB")print(f"Free memory      : {free / 1024**3:.1f} GiB")print(f"bfloat16 support : {torch.cuda.is_bf16_supported()}  (T4 -> False, falls back to fp16)")print(f"torch            : {torch.__version__}")!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 3. Clone the repositoryReplace `YOUR_USERNAME` with your GitHub account.

In [ ]:
import os, sysREPO_URL = "https://github.com/YOUR_USERNAME/llm-personal-finetuning.git"WORKDIR  = "/kaggle/working/llm-personal-finetuning"if not os.path.exists(WORKDIR):    !git clone --depth 1 $REPO_URL $WORKDIRelse:    !cd $WORKDIR && git pull --ff-onlyos.chdir(WORKDIR)sys.path.insert(0, WORKDIR)print("cwd:", os.getcwd())!git rev-parse --short HEAD

## 4. DatasetThe personal dataset is **not** committed to the repo. Attach it as a private KaggleDataset and point `SOURCE` at it. The cell copies it to `data/raw/personal_dataset.jsonl`and prints counts only - never contents.

In [ ]:
import shutilfrom pathlib import Path# Adjust to wherever your attached Kaggle Dataset landed.SOURCE = Path("/kaggle/input/personal-qa-dataset/personal_dataset.jsonl")TARGET = Path("data/raw/personal_dataset.jsonl")if SOURCE.exists():    TARGET.parent.mkdir(parents=True, exist_ok=True)    shutil.copy(SOURCE, TARGET)    print(f"copied {SOURCE} -> {TARGET}")elif TARGET.exists():    print(f"using dataset already in the repo: {TARGET}")else:    raise FileNotFoundError(        f"No dataset at {SOURCE} or {TARGET}. Attach your private Kaggle Dataset "        "(Add Data -> your dataset) and update SOURCE above."    )print("lines:", sum(1 for line in TARGET.open() if line.strip()))

## 5. Validate the dataset schema

In [ ]:
!python scripts/validate_dataset.py --strict

## 6. Deduplicate, detect leakage, and splitRecords are grouped by instruction similarity *before* splitting, so rewordedvariants of the same question cannot straddle train and test. `--fail-on-leakage`stops the notebook if any held-out question also appears in training.

In [ ]:
!python scripts/prepare_dataset.py --config configs/qlora.yaml --fail-on-leakage

In [ ]:
import jsonreport = json.load(open("data/reports/leakage_report.json"))print("split sizes    :", report["split"]["split_sizes"])print("actual ratios  :", report["split"]["actual_ratios"])print("groups         :", report["split"]["total_groups"])print("max overlap    :", report["leakage"]["max_overlap_rate"])print("leakage free   :", report["leakage"]["leakage_free"])

## 7. Inspect the model architectureConfirm the LoRA target modules against the *real* module tree rather thanassuming names from a tutorial.

In [ ]:
from transformers import AutoConfigfrom src.training.config import load_configconfig = load_config("configs/qlora.yaml")model_config = AutoConfig.from_pretrained(config.model_name)print("architecture :", model_config.architectures)print("model_type   :", model_config.model_type)print("layers       :", model_config.num_hidden_layers)print("hidden size  :", model_config.hidden_size)print("attn heads   :", model_config.num_attention_heads,      "| kv heads:", model_config.num_key_value_heads)

## 8. TrainCheckpoints, history, run metadata and the loss curve land in `artifacts/training/`.

In [ ]:
!python scripts/train_qlora.py --config configs/qlora.yaml

## 9. Training curve

In [ ]:
import jsonfrom IPython.display import Image, displayhistory = json.load(open("artifacts/training/training_history.json"))["history"]for entry in history:    if "validation_loss" in entry:        print(f"step {entry['step']:>5}  epoch {entry.get('epoch', 0):>5.2f}  "              f"train {entry.get('train_loss', float('nan')):.4f}  "              f"val {entry['validation_loss']:.4f}")display(Image("artifacts/training/loss_curve.png"))

## 10. Full evaluationBase vs fine-tuned on the untouched test set: perplexity, task metrics,generalization, catastrophic forgetting, final leakage check, and the promotion gate.This loads each model in turn (never both at once) to stay inside 16 GB.

In [ ]:
!python scripts/evaluate.py \    --config configs/qlora.yaml \    --eval-config configs/evaluation.yaml \    --adapter-path artifacts/training/adapter

## 11. Read the report

In [ ]:
from IPython.display import Markdown, displaydisplay(Markdown(open("artifacts/evaluation/final_report.md").read()))

In [ ]:
from IPython.display import Image, displayfor name in [    "training_vs_validation_loss",    "train_validation_gap",    "perplexity_comparison",    "base_vs_finetuned_task_score",    "generalization_comparison",    "catastrophic_forgetting_comparison",]:    path = f"artifacts/evaluation/plots/{name}.png"    try:        display(Image(path))    except FileNotFoundError:        print("missing:", path)

## 12. Spot-check some answers side by side

In [ ]:
!python scripts/compare_models.py \    --config configs/qlora.yaml \    --adapter-path artifacts/training/adapter \    --from-split data/eval/generalization.jsonl \    --limit 8

## 13. Optionally publish to the Hugging Face HubRuns **only** if the gate verdict is `PASS`. Requires a Kaggle secret named`HF_TOKEN` (Add-ons -> Secrets). The token is read into an environment variableand never printed.Only the adapter, tokenizer, model card, and aggregate evaluation reports areuploaded. The personal dataset is never uploaded.

In [ ]:
import jsonimport osHUB_MODEL_ID = "YOUR_USERNAME/deepseek-personal-qlora"  # <- change meverdict = json.load(open("artifacts/evaluation/gate.json"))["verdict"]print("gate verdict:", verdict)if verdict != "PASS":    print("Not publishing: the gate did not return PASS. Adjust configs/qlora.yaml and retrain.")else:    try:        from kaggle_secrets import UserSecretsClient        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")        print("HF_TOKEN loaded from Kaggle secrets")    except Exception as exc:        print("Could not load HF_TOKEN:", type(exc).__name__)    if os.environ.get("HF_TOKEN"):        !python scripts/push_to_hub.py \            --config configs/qlora.yaml \            --adapter-path artifacts/training/adapter \            --hub-model-id $HUB_MODEL_ID \            --require-pass    else:        print("HF_TOKEN not set - skipping publish.")

## 14. Download the artifactsKaggle keeps `/kaggle/working` after the session ends, but zip it up so you canpull the adapter and reports down in one click from the notebook output panel.

In [ ]:
!cd /kaggle/working && zip -qr adapter_and_reports.zip \    llm-personal-finetuning/artifacts/training/adapter \    llm-personal-finetuning/artifacts/training/training_history.json \    llm-personal-finetuning/artifacts/training/run_metadata.json \    llm-personal-finetuning/artifacts/evaluation \    llm-personal-finetuning/data/reports!ls -lh /kaggle/working/adapter_and_reports.zip

## Next experimentFree GPU quota is limited (~30 GPU-hours/week), so run experiments **one at a time**rather than sweeping. Swap the config and re-run from section 8:```!python scripts/train_qlora.py --config configs/experiments/exp_a_r8.yaml!python scripts/train_qlora.py --config configs/experiments/exp_b_r16.yaml!python scripts/train_qlora.py --config configs/experiments/exp_c_r32.yaml```Each experiment writes to its own `artifacts/experiments/<run_name>/` directory,so runs never overwrite each other.